# Split-strategy ablation -- all models

Two real notebooks merged into one file for a single, clean read-through:

1. **Part A** (below) -- SVM / Random Forest / VGG-16 / ResNet-18 under random split vs. CD-HIT-grouped split at 0.95 and 0.97 identity (feeds Supplementary Table S12 and the main leakage-delta tables).
2. **Part B** -- a small, focused ResNet-18-only follow-up (9 fits total) filling in the two split methodologies Part A's own ResNet-18 run doesn't cover: an ordinary random split (Supplementary Table S13) and the dataset's built-in MinHash near-duplicate clusters as the grouping variable (Supplementary Table S14).

Neither part ever used a GaussianNoise layer on ResNet-18's input (verified: 0 removals needed in Part A, 0 in Part B).

## Part A -- SVM / RF / VGG-16 / ResNet-18, random vs. CD-HIT-grouped split

# T23 — Leakage-delta rerun with the corrected fragment-sampling convention

**Why this notebook exists.** Two reviewers independently caught the same bug from two angles: Table 3
(the main fragmentation sweep, from Fragsweep/Nineseed) and Supplementary Table S9 (the random-split vs
CD-HIT-95%/97% leakage-delta comparison, from T23) report different VGG-16 macro-F1 at 300 bp on what
is supposedly the identical CD-HIT-97% split, same seeds, same clusters -- 0.646 in Table 3 versus
0.740 in Table S9. The cause is a one-character difference in how the two notebooks drew each random
fragment's start position: Fragsweep/Nineseed used `rng.integers(0, len(seq) - L + 1)` (includes every valid start
position), T23 used `rng.integers(0, len(seq) - L)` (silently excludes the last valid start position).
That one-character difference changes how each call consumes the shared NumPy random-number stream, so
the same seed produces a different sequence of sampled fragment windows in the two notebooks, even
though the underlying cluster-level train/test assignment is identical. The previous draft's response
to this was to report both numbers and call the discrepancy itself a finding about CNN sensitivity --
a reviewer correctly rejected that as rationalizing a reproducibility bug rather than fixing it.

**What this notebook does.** It is T23 with exactly one change: `fragment()` now uses the same
`rng.integers(0, len(seq) - L + 1)` convention as Fragsweep/Nineseed/T20/Ampmcnemar/Pralone, everywhere in this notebook
(both the random-split arm and the CD-HIT-grouped arm, at both 95% and 97% identity) -- not just the
one cell the discrepancy was noticed in, so every number in Supplementary Table S9 is internally
consistent with Table 3 and with itself. Everything else (CD-HIT commands, split logic, model configs,
CNN architectures and training loop) is unchanged from T23, so the *comparison* T23 was designed to make
(random split vs. leakage-controlled split, at short and full length) is preserved -- only the sampling
bug is fixed.

**After this run:** replace Supplementary Table S9 with these numbers, and re-check the RT-vs-PR+RT
question and the VGG-16-at-300bp discrepancy described above -- both should resolve once every table in
the paper is drawing fragments the same way.

**This notebook is written to run unattended end to end**, same as T23. The full grid (2 identity
levels x 2 split arms x 3 lengths x 3 seeds x 3 CNNs = 108 CNN fits) is the most expensive run in this
project, realistically several hours of GPU time -- read Section 2's `FAST_MODE` / `RUN_DEEP_MODELS`
knobs before you commit and walk away.

**GPU needed.** Yes, for Section 8 (the three CNNs). Sections 1-7 (CD-HIT clustering plus SVM and
random forest) run fine on CPU alone.

**Dataset.** All notebooks below read the *same* dataset you already used for the main experiments and the 5-model fragment sweep:

`/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv`

with columns `sequence`, `subtype`, `cluster` (the MinHash cluster id). Attach that Kaggle Dataset to the notebook the same way you did for the fragment-test notebook (Add Data -> Your Datasets -> subtype-classification). **No new download is needed** for this file. Where a test needs something extra (a reference genome, an external tool), that is called out explicitly at the top of the notebook, with the exact cell that fetches or installs it -- it downloads automatically inside the notebook, you do not need to upload anything yourself.


**Kaggle session budget.** Kaggle notebook sessions cap out at **12 hours** and (for GPU) a **~30 GPU-hours/week** quota. Each notebook below states whether it needs a GPU and gives a rough runtime estimate at the default settings, plus a `FAST_MODE` toggle to shrink seeds/epochs/lengths if you are short on quota. If a run is approaching the 12 h wall, results are saved to `/kaggle/working` incrementally (per seed / per length) so a partial run is still usable -- just re-run with the remaining settings and concatenate the CSVs.


### 1. Imports and install CD-HIT (same as T2 -- installed here, nothing to download yourself)

In [ ]:

import warnings; warnings.filterwarnings("ignore")
import os, subprocess, shutil, random
import numpy as np, pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, balanced_accuracy_score

OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

if shutil.which("cd-hit-est") is None:
    print("cd-hit-est not found, installing...")
    subprocess.run("conda install -y -c bioconda cd-hit 2>/dev/null || true", shell=True)
    if shutil.which("cd-hit-est") is None:
        subprocess.run(
            "wget -q https://github.com/weizhongli/cdhit/releases/download/V4.8.1/cd-hit-v4.8.1-2019-0228.tar.gz "
            "&& tar xzf cd-hit-v4.8.1-2019-0228.tar.gz && cd cd-hit-v4.8.1-2019-0228 && make -j4 "
            "&& cp cd-hit-est /usr/local/bin/ && cp cd-hit /usr/local/bin/", shell=True)
print("cd-hit-est available:", shutil.which("cd-hit-est") is not None)


### 2. Config -- READ THIS before you Run All and walk away

`RUN_DEEP_MODELS = True` plus `FAST_MODE = False` runs the full grid: 108 CNN fits, realistically 4-9 hours of GPU time on top of the CPU-only classical section. If that does not fit your remaining GPU budget, set `FAST_MODE = True` now (2 seeds, one identity level, halves the grid) or `RUN_DEEP_MODELS = False` (classical only, no GPU needed at all, finishes in under an hour) -- both are safe to leave unattended, they just produce a smaller result.

In [ ]:

FAST_MODE       = False          # True: 2 seeds, only 97% identity (roughly halves total runtime)
SEEDS           = [0, 1] if FAST_MODE else [0, 1, 2]
IDENTITIES      = [0.97] if FAST_MODE else [0.95, 0.97]
RUN_DEEP_MODELS = True            # False: classical-only (SVM/RF), no GPU needed, finishes in <1h
LENGTHS         = [300, 500, None]   # None = full length
MAX_TFIDF_FEATS = 512
DEEP_MODELS     = ["resnet18", "inception"] if FAST_MODE else ["resnet18", "vgg16", "inception"]


### 3. Load data and write FASTA for CD-HIT

In [ ]:

DATA_PATH = "/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv"
df = pd.read_csv(DATA_PATH).reset_index(drop=True)

le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
n_cls = len(le.classes_)
X_all = df["sequence"].values
n = len(df)
print(f"{n} sequences, {n_cls} classes")

with open(f"{OUT}/pol_t23.fasta", "w") as f:
    for i, s in enumerate(X_all):
        f.write(f">{i}\n{s}\n")


### 4. Run CD-HIT at each identity level (same as T2/T16) and parse cluster assignments

In [ ]:

def run_cdhit(identity, word_size):
    out = f"{OUT}/pol_t23_cdhit_{int(identity*100)}"
    cmd = f"cd-hit-est -i {OUT}/pol_t23.fasta -o {out} -c {identity} -n {word_size} -M 16000 -T 4 -d 0"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return out, r.returncode

def parse_cdhit_clusters(clstr_path, n_seqs):
    labels = np.full(n_seqs, -1, dtype=int)
    cid = -1
    with open(clstr_path) as f:
        for line in f:
            if line.startswith(">Cluster"):
                cid += 1
            else:
                idx = int(line.split(">")[1].split("...")[0])
                labels[idx] = cid
    assert (labels >= 0).all(), "some sequences never got a cluster id -- CD-HIT output does not match the input FASTA"
    return labels

WORD_SIZE = {0.95: 10, 0.97: 10}   # same word size T2 used, kept identical for consistency
cdhit_groups = {}
for identity in IDENTITIES:
    out, rc = run_cdhit(identity, WORD_SIZE[identity])
    assert rc == 0 and os.path.exists(out + ".clstr"), f"CD-HIT failed at identity={identity} (returncode={rc})"
    labs = parse_cdhit_clusters(out + ".clstr", n)
    cdhit_groups[identity] = labs
    print(f"CD-HIT {identity*100:.0f}% identity: {len(set(labs))} clusters, "
          f"largest = {pd.Series(labs).value_counts().iloc[0]} sequences")


### 5. Fragmentation helper (same convention as the other notebooks)

In [ ]:

def fragment(seq, L, rng):
    if L is None or len(seq) <= L:
        return seq
    start = rng.integers(0, len(seq) - L + 1)   # <-- THE FIX: was `len(seq) - L` in T23 (excludes the
    return seq[start:start+L]                   #     last valid start position); now matches Fragsweep/Nineseed/T20/Ampmcnemar/Pralone.


### 6. Split helpers -- one random arm, one CD-HIT-grouped arm per identity level, each carving its own validation fold for the CNNs

In [ ]:

def split_random(X, y, seed):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    tr_idx, te_idx = next(skf.split(X, y))
    skf2 = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed)
    tr2, va2 = next(skf2.split(X[tr_idx], y[tr_idx]))
    return tr_idx[tr2], tr_idx[va2], te_idx

def split_grouped(X, y, grp, seed):
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(X, y, grp))
    s2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(s2.split(X[trv], y[trv], grp[trv]))
    return trv[tr_rel], trv[va_rel], te


### 7. Classical models (SVM, RF) at every length, both split arms -- CPU only

In [ ]:

def fit_tfidf(seq_tr, seq_va, seq_te):
    vec = TfidfVectorizer(analyzer="char", ngram_range=(6,6), max_features=MAX_TFIDF_FEATS, lowercase=False)
    Xtr = vec.fit_transform(seq_tr); Xva = vec.transform(seq_va); Xte = vec.transform(seq_te)
    return Xtr, Xva, Xte

def fit_score_classical(Xtr, ytr, Xte, yte, model_name, seed):
    if model_name == "svm":
        clf = LinearSVC(class_weight="balanced", random_state=seed, max_iter=20000)
    elif model_name == "rf":
        clf = RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=seed, n_jobs=-1)
    else:
        raise ValueError(model_name)
    clf.fit(Xtr, ytr)
    pred = clf.predict(Xte)
    return f1_score(yte, pred, average="macro"), balanced_accuracy_score(yte, pred)

CLASSICAL = ["svm", "rf"]
rows = []
splits_cache = {}   # (tag, seed, arm, identity) -> (tr, va, te) index arrays, reused by the CNN section below

for L in LENGTHS:
    tag = L if L else "full"
    for seed in SEEDS:
        rng = np.random.default_rng(seed)
        seqs = np.array([fragment(s, L, rng) for s in X_all])

        tr, va, te = split_random(seqs, y_all, seed)
        splits_cache[(tag, seed, "random", None)] = (tr, va, te)
        Xtr, Xva, Xte = fit_tfidf(seqs[tr], seqs[va], seqs[te])
        for m in CLASSICAL:
            f1, bal = fit_score_classical(Xtr, y_all[tr], Xte, y_all[te], m, seed)
            rows.append(dict(length=tag, seed=seed, arm="random", identity=None, model=m, f1=f1, bal_acc=bal))

        for identity, groups in cdhit_groups.items():
            tr, va, te = split_grouped(seqs, y_all, groups, seed)
            splits_cache[(tag, seed, "cdhit_grouped", identity)] = (tr, va, te)
            Xtr, Xva, Xte = fit_tfidf(seqs[tr], seqs[va], seqs[te])
            for m in CLASSICAL:
                f1, bal = fit_score_classical(Xtr, y_all[tr], Xte, y_all[te], m, seed)
                rows.append(dict(length=tag, seed=seed, arm="cdhit_grouped", identity=identity,
                                  model=m, f1=f1, bal_acc=bal))
        pd.DataFrame(rows).to_csv(f"{OUT}/t23_cdhit_split_classical_raw.csv", index=False)  # incremental save
        print(f"[classical] length={tag} seed={seed} done")

t23_classical = pd.DataFrame(rows)
t23_classical.groupby(["length","arm","identity","model"])[["f1","bal_acc"]].agg(["mean","std"]).round(4)


### 8. CNNs at every length, both split arms -- needs GPU. Same architectures and training loop as T1, so results are directly comparable to Table 7.

In [ ]:

if RUN_DEEP_MODELS:
    import tensorflow as tf
    from tensorflow.keras import layers, Model, Input

    def conv_bn_relu(x, filters, kernel=5, stride=1):
        x = layers.Conv1D(filters, kernel, strides=stride, padding="same")(x)
        x = layers.BatchNormalization()(x)
        return layers.Activation("relu")(x)

    def build_vgg16_1d(input_len, n_classes):
        inp = Input(shape=(input_len, 1)); x = inp
        for filters, n_convs in [(32,2),(64,2),(128,3),(256,3),(256,3)]:
            for _ in range(n_convs):
                x = conv_bn_relu(x, filters, 3)
            x = layers.MaxPooling1D(2, padding="same")(x)
        x = layers.GlobalAveragePooling1D()(x); x = layers.Dropout(0.3)(x)
        out = layers.Dense(n_classes, activation="softmax")(x)
        return Model(inp, out, name="vgg16_1d")

    class StochasticDepth(layers.Layer):
        def __init__(self, p=0.85, **kw):
            super().__init__(**kw); self.p = p
        def call(self, x, training=None):
            if not training:
                return x
            r = tf.random.uniform([tf.shape(x)[0], 1, 1])
            return x * tf.floor(self.p + r) / self.p
        def get_config(self):
            return {**super().get_config(), "p": self.p}

    def res_block(x, filters, stride=1):
        shortcut = x
        y = conv_bn_relu(x, filters, 3, stride)
        y = layers.Conv1D(filters, 3, padding="same")(y); y = layers.BatchNormalization()(y)
        if shortcut.shape[-1] != filters or stride != 1:
            shortcut = layers.Conv1D(filters, 1, strides=stride, padding="same")(shortcut)
            shortcut = layers.BatchNormalization()(shortcut)
        y = StochasticDepth(0.85)(y)
        return layers.Activation("relu")(layers.Add()([shortcut, y]))

    def build_resnet18_1d(input_len, n_classes):
        inp = Input(shape=(input_len, 1))
        x = conv_bn_relu(inp, 32, 7, 2); x = layers.MaxPooling1D(3, strides=2, padding="same")(x)
        for filters, stride in [(32,1),(32,1),(64,2),(64,1),(128,2),(128,1),(256,2),(256,1)]:
            x = res_block(x, filters, stride)
        x = layers.GlobalAveragePooling1D()(x); x = layers.Dropout(0.3)(x)
        out = layers.Dense(n_classes, activation="softmax")(x)
        return Model(inp, out, name="resnet18_1d")

    def inception_module(x, f1x1, f3x3, f5x5, fpool):
        b1 = conv_bn_relu(x, f1x1, 1); b2 = conv_bn_relu(x, f3x3, 3); b3 = conv_bn_relu(x, f5x5, 5)
        b4 = layers.MaxPooling1D(3, strides=1, padding="same")(x); b4 = conv_bn_relu(b4, fpool, 1)
        return layers.Concatenate()([b1, b2, b3, b4])

    def build_inception_1d(input_len, n_classes):
        inp = Input(shape=(input_len, 1))
        x = conv_bn_relu(inp, 32, 7, 2); x = layers.MaxPooling1D(3, strides=2, padding="same")(x)
        x = inception_module(x, 16, 24, 8, 8); x = inception_module(x, 32, 40, 16, 16)
        x = layers.MaxPooling1D(2, padding="same")(x); x = inception_module(x, 40, 48, 20, 20)
        x = layers.GlobalAveragePooling1D()(x); x = layers.Dropout(0.3)(x)
        out = layers.Dense(n_classes, activation="softmax")(x)
        return Model(inp, out, name="inception_v1_1d")

    BUILDERS = {"vgg16": build_vgg16_1d, "resnet18": build_resnet18_1d, "inception": build_inception_1d}

    def run_deep(model_name, Xtr, ytr, Xva, yva, Xte, yte, seed, epochs=60, batch=48):
        tf.random.set_seed(seed)
        Xtr_d = np.asarray(Xtr.todense() if hasattr(Xtr, "todense") else Xtr, dtype=np.float32)[..., None]
        Xva_d = np.asarray(Xva.todense() if hasattr(Xva, "todense") else Xva, dtype=np.float32)[..., None]
        Xte_d = np.asarray(Xte.todense() if hasattr(Xte, "todense") else Xte, dtype=np.float32)[..., None]
        mu, sd = Xtr_d.mean(), Xtr_d.std() + 1e-8
        Xtr_d = (Xtr_d - mu) / sd; Xva_d = (Xva_d - mu) / sd; Xte_d = (Xte_d - mu) / sd
        model = BUILDERS[model_name](Xtr_d.shape[1], n_cls)
        cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))
        model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                       loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
        es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
        model.fit(Xtr_d, ytr, validation_data=(Xva_d, yva), epochs=epochs, batch_size=batch,
                  class_weight=cw, callbacks=[es], verbose=0)
        pred = np.argmax(model.predict(Xte_d, verbose=0), axis=1)
        f1 = f1_score(yte, pred, average="macro"); bal = balanced_accuracy_score(yte, pred)
        tf.keras.backend.clear_session()
        return f1, bal

    deep_rows = []
    for (tag, seed, arm, identity), (tr, va, te) in splits_cache.items():
        L = None if tag == "full" else tag
        rng = np.random.default_rng(seed)
        seqs = np.array([fragment(s, L, rng) for s in X_all])
        Xtr, Xva, Xte = fit_tfidf(seqs[tr], seqs[va], seqs[te])
        for m in DEEP_MODELS:
            f1, bal = run_deep(m, Xtr, y_all[tr], Xva, y_all[va], Xte, y_all[te], seed)
            deep_rows.append(dict(length=tag, seed=seed, arm=arm, identity=identity, model=m, f1=f1, bal_acc=bal))
        pd.DataFrame(deep_rows).to_csv(f"{OUT}/t23_cdhit_split_deep_raw.csv", index=False)  # incremental save
        print(f"[deep] length={tag} seed={seed} arm={arm} identity={identity} done")

    t23_deep = pd.DataFrame(deep_rows)
    t23_all = pd.concat([t23_classical, t23_deep], ignore_index=True)
else:
    print("RUN_DEEP_MODELS is False -- skipped. t23_all is classical-only.")
    t23_all = t23_classical

t23_all.to_csv(f"{OUT}/t23_all_raw.csv", index=False)


### 9. Summary table matching the format of Table 7 (delta = random minus cdhit_grouped)

In [ ]:

summary = (t23_all.groupby(["length","arm","identity","model"])["f1"].mean()
           .reset_index().pivot_table(index=["length","model"], columns=["arm","identity"], values="f1"))
summary.to_csv(f"{OUT}/t23_summary_table.csv")
print(summary.round(4))
print("""
Read this table by row (one length x one model). A delta near the T1/Table 7 full-length delta
(under 0.01) at 300 and 500 bp too means leakage genuinely doesn't matter much at this task even
under fragmentation. A delta that GROWS at short lengths means the honest leakage story is
different from what Table 7 alone suggested, and Table 7 / Section 4.1 / the Introduction need to
be rewritten around this new, more complete result.
""")


In [ ]:
!cd /kaggle/working && zip -r t23_cdhit_split_outputs.zip t23_all_raw.csv t23_summary_table.csv t23_cdhit_split_classical_raw.csv

## Part B -- ResNet-18 only: random split (S13) and MinHash-grouped split (S14) follow-up

# ResNet-18 — Supplementary Tables S13 & S14 follow-up (random split and MinHash-grouped split, GaussianNoise removed)

**Why this notebook exists.** The main no-GaussianNoise rerun (`ResNet18_full_reverification_no_gaussian_noise.ipynb`)
reruns ResNet-18 under the CD-HIT-97%-grouped split used everywhere else in this study. Two supplementary tables
need ResNet-18 numbers under two *other* split methodologies that notebook never touches:

- **Supplementary Table S13** (random-split leakage ablation) needs ResNet-18 trained with an ordinary random
  split that ignores clustering entirely, at 300 bp and full length, so it can be set against the already-known
  CD-HIT-97% value from the main rerun (300 bp: 0.753, full length: 0.976 — Supplementary Table S4).
- **Supplementary Table S14** (MinHash-grouped split versus CD-HIT-97% split) needs ResNet-18 trained with the
  dataset's built-in MinHash near-duplicate clusters (the `cluster` column in `prepared_sequences_clean.csv`,
  Jaccard threshold 0.90) as the grouping variable, at full length only, again set against the same CD-HIT-97%
  full-length value (0.976).

This notebook is deliberately small: **9 ResNet-18 fits in total** (6 for S13: 300 bp + full length x 3 seeds;
3 for S14: full length x 3 seeds), all using the exact same architecture, training config and TF-IDF 6-mer
encoding as the main no-GaussianNoise rerun — only the train/test split logic differs. No GaussianNoise layer
anywhere, consistent with the rest of this study.

**GPU needed.** Yes. **Runtime.** 9 fits at roughly the same per-fit cost as the main rerun (10-65 min each
depending on early stopping) — realistically 1.5-10 GPU-hours, comfortably inside one Kaggle session.

**How to run.** Edit `DATA_PATH` in Section 2 if your dataset path differs, then Run All. Each section saves
incrementally and skips any `(split, length, seed)` combination it finds already in its own CSV, so a restart
mid-run just picks up where it left off — no separate resume-bridge cells needed for a notebook this small.

**What to send back.** Two CSVs: `resnet18_S13_randomsplit_raw.csv` and `resnet18_S14_minhashsplit_raw.csv`,
both written to `/kaggle/working/`.

### 1. Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, random
import numpy as np, pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, matthews_corrcoef, balanced_accuracy_score, accuracy_score

import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization, ReLU, MaxPooling1D,
                                      SpatialDropout1D, Add, GlobalAveragePooling1D, Dropout,
                                      Dense, Layer)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

print("TF:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

### 2. Config

In [ ]:
DATA_PATH = "/kaggle/input/datasets/yashrajtheking/msc-work-final/prepared_sequences_clean.csv"  # <-- EDIT if your path differs

SEEDS          = [0, 1, 2]       # same three seeds used throughout this study
LENGTHS_S13    = [300, None]     # S13: 300 bp and full length; None = full length
EPOCHS         = 120
BATCH          = 48
OUT            = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

### 3. Load data

In [ ]:
df = pd.read_csv(DATA_PATH)
le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
CLASSES = list(le.classes_); n_cls = len(CLASSES)
X_all = df["sequence"].values
print(f"{len(df)} seqs | {n_cls} classes: {CLASSES}")

assert "cluster" in df.columns, "expected a MinHash 'cluster' column in prepared_sequences_clean.csv"
grp_minhash = df["cluster"].values
print(f"MinHash near-duplicate clusters (Jaccard >= 0.90): {len(set(grp_minhash))} clusters, "
      f"largest = {pd.Series(grp_minhash).value_counts().iloc[0]} sequences")

### 4. Shared helpers: splits, fragmentation, TF-IDF 6-mer encoding, metrics

Fragment sampling uses `rng.integers(0, len(seq) - L + 1)` — the convention used throughout this study (includes every valid start position) — matching the main no-GaussianNoise rerun exactly.

In [ ]:
def split_random(X, y, seed):
    # ordinary random split: completely ignores any clustering/grouping -- this is the
    # "random split" arm needed for Supplementary Table S13
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    tr_idx, te_idx = next(skf.split(X, y))
    skf2 = StratifiedKFold(n_splits=4, shuffle=True, random_state=seed)
    tr2, va2 = next(skf2.split(X[tr_idx], y[tr_idx]))
    return tr_idx[tr2], tr_idx[va2], te_idx

def split_grouped(X, y, grp, seed):
    # cluster-disjoint split for any grouping variable -- used here with the MinHash
    # `cluster` column for Supplementary Table S14 (the CD-HIT-97% arm itself is not
    # rerun here; its value is already known from the main no-GaussianNoise rerun)
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(X, y, grp))
    s2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(s2.split(X[trv], y[trv], grp[trv]))
    return trv[tr_rel], trv[va_rel], te

def fragment(seqs, L, rng):
    if L is None: return list(seqs)
    out = []
    for s in seqs:
        if len(s) <= L: out.append(s)
        else:
            st = int(rng.integers(0, len(s) - L + 1))
            out.append(s[st:st+L])
    return out

def kmers(seq, k=6): return [seq[i:i+k] for i in range(len(seq)-k+1)]

def make_tfidf(train, k=6, max_features=512):
    vec = TfidfVectorizer(max_features=max_features)
    vec.fit([" ".join(kmers(s,k)) for s in train])
    return lambda seqs: vec.transform([" ".join(kmers(s,k)) for s in seqs]).toarray().astype(np.float32)

def score(y_true, y_pred):
    return dict(f1=round(f1_score(y_true, y_pred, average="macro"),4),
                mcc=round(matthews_corrcoef(y_true, y_pred),4),
                bal_acc=round(balanced_accuracy_score(y_true, y_pred),4),
                acc=round(accuracy_score(y_true, y_pred),4))

### 5. ResNet-18 (1D), GaussianNoise removed — identical architecture and training config to the main no-GaussianNoise rerun

In [ ]:
class StochasticDepth(Layer):
    def __init__(self, p=0.85, **kw): super().__init__(**kw); self.p = p
    def call(self, x, training=None):
        if not training: return x
        r = tf.random.uniform([tf.shape(x)[0],1,1]); return x*tf.floor(self.p+r)/self.p
    def get_config(self): return {**super().get_config(), "p": self.p}

def res_block(x, f, stride=1, sp=0.85, dr=0.1):
    sc = x
    x = Conv1D(f,3,strides=stride,padding="same",use_bias=False)(x)
    x = ReLU()(BatchNormalization()(x)); x = SpatialDropout1D(dr)(x)
    x = BatchNormalization()(Conv1D(f,3,padding="same",use_bias=False)(x))
    if stride!=1 or sc.shape[-1]!=f:
        sc = BatchNormalization()(Conv1D(f,1,strides=stride,padding="same",use_bias=False)(sc))
    return ReLU()(Add()([StochasticDepth(sp)(x), sc]))

def build_resnet18_no_noise(L, n):
    inp = Input((L,1))
    x = inp                                              # <-- no GaussianNoise layer, same as the main rerun
    x = ReLU()(BatchNormalization()(Conv1D(32,7,strides=2,padding="same",use_bias=False)(x)))
    x = MaxPooling1D(3,strides=2,padding="same")(x)
    for ff,sp in [(32,.95),(32,.92)]:   x = res_block(x,ff,1,sp)
    for ff,sp in [(64,.88),(64,.85)]:   x = res_block(x,ff,2 if ff!=x.shape[-1] else 1,sp)
    for ff,sp in [(128,.80),(128,.78)]: x = res_block(x,ff,2 if ff!=x.shape[-1] else 1,sp)
    for ff,sp in [(256,.75),(256,.72)]: x = res_block(x,ff,2 if ff!=x.shape[-1] else 1,sp)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    m = Model(inp, Dense(n,activation="softmax")(x))
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

def train_resnet(ftr, fva, fte, ytr, yva, cw):
    sc = StandardScaler()                                 # fit on the training fold only
    a = sc.fit_transform(ftr)[...,None]; b = sc.transform(fva)[...,None]; c = sc.transform(fte)[...,None]
    tf.keras.backend.clear_session()
    m = build_resnet18_no_noise(a.shape[1], n_cls)
    m.fit(a, to_categorical(ytr,n_cls), validation_data=(b,to_categorical(yva,n_cls)),
          epochs=EPOCHS, batch_size=BATCH, class_weight=cw, shuffle=True, verbose=0,
          callbacks=[EarlyStopping("val_loss",patience=15,restore_best_weights=True),
                     ReduceLROnPlateau("val_loss",factor=0.5,patience=6,min_lr=1e-6)])
    return m.predict(c, verbose=0).argmax(1)

## Section S13 — Random split (no grouping), 300 bp and full length, 3 seeds

Pairs against the already-known CD-HIT-97% values (300 bp: 0.753, full length: 0.976) from Supplementary Table S4 of the main no-GaussianNoise rerun.

In [ ]:
S13_RAW = f"{OUT}/resnet18_S13_randomsplit_raw.csv"
S13_rows = pd.read_csv(S13_RAW).to_dict("records") if os.path.exists(S13_RAW) else []
S13_done = {(r["length"], r["seed"]) for r in S13_rows}
print(f"[S13] resuming: {len(S13_done)} (length, seed) combinations already done.")

for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tr, va, te = split_random(X_all, y_all, seed)
    ytr, yva, yte = y_all[tr], y_all[va], y_all[te]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))

    for L in LENGTHS_S13:
        label = "full" if L is None else str(L)
        if (label, seed) in S13_done:
            print(f"[S13][skip] length={label} seed={seed} already done"); continue

        rng = np.random.default_rng(seed*100 + (L or 9999) + 13000)
        Ftr = fragment(X_all[tr], L, rng); Fva = fragment(X_all[va], L, rng); Fte = fragment(X_all[te], L, rng)

        tfidf = make_tfidf(Ftr)
        Ttr, Tva, Tte = tfidf(Ftr), tfidf(Fva), tfidf(Fte)

        pred = train_resnet(Ttr, Tva, Tte, ytr, yva, cw)

        S13_rows.append({"split":"random","length":label,"length_num":(L or 3000),"seed":seed,
                          **score(yte, pred)})
        pd.DataFrame(S13_rows).to_csv(S13_RAW, index=False)
        print(f"[S13][seed {seed}] L={label:>4}  random-split f1={S13_rows[-1]['f1']}  bal_acc={S13_rows[-1]['bal_acc']}")

S13_summ = (pd.DataFrame(S13_rows).groupby(["length","length_num"])[["f1","mcc","bal_acc","acc"]]
              .agg(["mean","std"]).round(4))
S13_summ.columns = [f"{m}_{s}" for m,s in S13_summ.columns]
S13_summ = S13_summ.reset_index().sort_values("length_num")
S13_summ.to_csv(f"{OUT}/resnet18_S13_randomsplit_summary.csv", index=False)
print(S13_summ.to_string(index=False))

## Section S14 — MinHash-grouped split, full length only, 3 seeds

Uses the dataset's built-in `cluster` column (MinHash near-duplicate grouping, Jaccard >= 0.90) as the grouping variable for `StratifiedGroupKFold`, otherwise identical to the CD-HIT-97% split logic. Pairs against the already-known CD-HIT-97% full-length value (0.976, Supplementary Table S4).

In [ ]:
S14_RAW = f"{OUT}/resnet18_S14_minhashsplit_raw.csv"
S14_rows = pd.read_csv(S14_RAW).to_dict("records") if os.path.exists(S14_RAW) else []
S14_done = {r["seed"] for r in S14_rows}
print(f"[S14] resuming: {len(S14_done)} seed(s) already done.")

for seed in SEEDS:
    if seed in S14_done:
        print(f"[S14][skip] seed={seed} already done"); continue

    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    tr, va, te = split_grouped(X_all, y_all, grp_minhash, seed)
    ytr, yva, yte = y_all[tr], y_all[va], y_all[te]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))

    Ftr, Fva, Fte = list(X_all[tr]), list(X_all[va]), list(X_all[te])   # full length -> no fragmentation
    tfidf = make_tfidf(Ftr)
    Ttr, Tva, Tte = tfidf(Ftr), tfidf(Fva), tfidf(Fte)

    pred = train_resnet(Ttr, Tva, Tte, ytr, yva, cw)

    S14_rows.append({"split":"minhash_grouped","length":"full","length_num":3000,"seed":seed,
                      **score(yte, pred)})
    pd.DataFrame(S14_rows).to_csv(S14_RAW, index=False)
    print(f"[S14][seed {seed}] minhash-grouped f1={S14_rows[-1]['f1']}  bal_acc={S14_rows[-1]['bal_acc']}")

S14_summ = pd.DataFrame(S14_rows)[["f1","mcc","bal_acc","acc"]].agg(["mean","std"]).round(4)
S14_summ.to_csv(f"{OUT}/resnet18_S14_minhashsplit_summary.csv")
print(S14_summ.to_string())

### 6. Zip outputs for download

In [ ]:
!cd /kaggle/working && zip -r resnet18_s13_s14_outputs.zip \
    resnet18_S13_randomsplit_raw.csv resnet18_S13_randomsplit_summary.csv \
    resnet18_S14_minhashsplit_raw.csv resnet18_S14_minhashsplit_summary.csv
print("zipped. Download resnet18_s13_s14_outputs.zip from the Kaggle Output tab, "
      "or just the four CSVs individually -- either works.")